Literature Download using Elsapy

Which Literature we could use

apikey = "3af51ecbe651ef826516793ab9bbed05"
Search Elsapy for Keywords

In [19]:
import time
import urllib.parse
import pandas as pd
from elsapy.elsclient import ElsClient
from elsapy.elsdoc import FullDoc

apikey = "3af51ecbe651ef826516793ab9bbed05"
client = ElsClient(apikey)

max_papers = 1000
batch_size = 25

# broad candidate search
query = """
TITLE-ABS-KEY(
    weld* AND
    (HSLA OR "low alloy steel" OR steel* OR alloy*) AND
    (experiment* OR "experimental procedure" OR "experimental setup" OR methods) AND
    ("heat input" OR "cooling rate" OR microstructure OR "yield strength" OR toughness)
)
"""

# category-based keywords for full-text relevance
keyword_groups = {
    "procedure": [
        "experimental procedure",
        "experimental setup",
        "materials and methods",
        "methodology",
        "sample preparation",
        "welding procedure",
        "process parameters",
        "experimental details",
        "procedure"
    ],
    "equipment": [
        "welding machine",
        "torch",
        "thermocouple",
        "scanning electron microscope",
        "sem",
        "eds",
        "microhardness",
        "hardness test",
        "tensile test",
        "tensile testing",
        "impact test",
        "charpy",
        "xrd"
    ],
    "process": [
        "heat input",
        "cooling rate",
        "current",
        "voltage",
        "travel speed",
        "welding speed",
        "shielding gas",
        "interpass temperature",
        "preheat",
        "post weld heat treatment",
        "filler metal",
        "weld metal",
        "base metal",
        "gmaw",
        "gtaw",
        "saw",
        "fcaw"
    ],
    "outputs": [
        "yield strength",
        "toughness",
        "microstructure",
        "hardness",
        "residual stress",
        "grain size",
        "phase transformation",
        "bainite",
        "martensite",
        "acicular ferrite",
        "carbides",
        "cct",
        "ttt"
    ]
}

print("Step 1: Starting paged search...")
t0 = time.time()

all_results = []
start = 0
total_available = None

encoded_query = urllib.parse.quote(query)
base_url = "https://api.elsevier.com/content/search/scopus"

while len(all_results) < max_papers:
    url = f"{base_url}?query={encoded_query}&count={batch_size}&start={start}"
    print(f"Requesting results {start} to {start + batch_size - 1}...")

    data = client.exec_request(url)
    search_results = data.get("search-results", {})
    entries = search_results.get("entry", [])

    if total_available is None:
        total_available = int(search_results.get("opensearch:totalResults", 0))
        print("Total available in Scopus:", total_available)

    if not entries:
        print("No more results returned.")
        break

    all_results.extend(entries)
    print(f"  Got {len(entries)} results, total collected: {len(all_results)}")

    if len(entries) < batch_size:
        print("Last page reached.")
        break

    start += batch_size
    if start >= total_available:
        print("Reached total available results.")
        break

all_results = all_results[:max_papers]

print(f"Step 1 done in {time.time() - t0:.2f} sec")
print("Requested:", max_papers)
print("Collected:", len(all_results))

print("Step 2: Building search dataframe...")
search_records = []
for r in all_results:
    search_records.append({
        "title": r.get("dc:title", ""),
        "pii": r.get("pii", ""),
        "doi": r.get("prism:doi", ""),
        "eid": r.get("eid", ""),
        "publication": r.get("prism:publicationName", ""),
        "coverDate": r.get("prism:coverDate", ""),
    })

search_df = pd.DataFrame(search_records)
print("Search dataframe size:", len(search_df))
display(search_df.head())

print("Step 3: Preparing identifiers...")

records_to_check = []
seen = set()

for _, row in search_df.iterrows():
    pii = str(row["pii"]).strip() if pd.notna(row["pii"]) else ""
    doi = str(row["doi"]).strip() if pd.notna(row["doi"]) else ""

    if pii:
        key = ("pii", pii)
        if key not in seen:
            seen.add(key)
            records_to_check.append({
                "id_type": "pii",
                "id_value": pii,
                "title": row["title"],
                "doi": doi,
            })
    elif doi:
        key = ("doi", doi)
        if key not in seen:
            seen.add(key)
            records_to_check.append({
                "id_type": "doi",
                "id_value": doi,
                "title": row["title"],
                "doi": doi,
            })

print("Total search results:", len(search_df))
print("Records with PII or DOI:", len(records_to_check))

def extract_text(raw_text):
    if isinstance(raw_text, str):
        return raw_text
    if isinstance(raw_text, dict):
        texts = []

        def recurse(x):
            if isinstance(x, str):
                texts.append(x)
            elif isinstance(x, dict):
                for v in x.values():
                    recurse(v)
            elif isinstance(x, list):
                for item in x:
                    recurse(item)

        recurse(raw_text)
        return " ".join(texts)
    return ""

def score_relevance(full_text, keyword_groups):
    matched_groups = {}

    for group_name, kws in keyword_groups.items():
        hits = [kw for kw in kws if kw.lower() in full_text]
        matched_groups[group_name] = hits

    group_hit_count = sum(len(v) > 0 for v in matched_groups.values())
    total_hits = sum(len(v) for v in matched_groups.values())

    return matched_groups, group_hit_count, total_hits

def flatten_matched_groups(matched_groups):
    parts = []
    for group_name, hits in matched_groups.items():
        if hits:
            parts.append(f"{group_name}: {', '.join(hits)}")
    return " | ".join(parts)

print("Step 4: Checking full text and content relevance...")
results = []

for i, rec in enumerate(records_to_check, 1):
    item_start = time.time()
    id_type = rec["id_type"]
    id_value = rec["id_value"]

    print(f"[{i}/{len(records_to_check)}] Checking {id_type.upper()}: {id_value}")

    try:
        if id_type == "pii":
            doc = FullDoc(sd_pii=id_value)
        else:
            doc = FullDoc(doi=id_value)
    except Exception as e:
        results.append({
            "id_type": id_type,
            "id_value": id_value,
            "title": rec["title"],
            "doi": rec["doi"],
            "openaccess": "",
            "full_text_available": False,
            "matched_keywords": "",
            "group_hit_count": 0,
            "total_keyword_hits": 0,
            "relevant_in_fulltext": False,
            "reason": f"init_failed: {e}",
        })
        print(f"  -> init failed in {time.time() - item_start:.2f} sec")
        continue

    if not doc.read(client):
        results.append({
            "id_type": id_type,
            "id_value": id_value,
            "title": rec["title"],
            "doi": rec["doi"],
            "openaccess": "",
            "full_text_available": False,
            "matched_keywords": "",
            "group_hit_count": 0,
            "total_keyword_hits": 0,
            "relevant_in_fulltext": False,
            "reason": "read_failed",
        })
        print(f"  -> read failed in {time.time() - item_start:.2f} sec")
        continue

    core = doc.data.get("coredata", {})
    raw = doc.data.get("originalText", "")
    objects = doc.data.get("objects", [])

    has_fulltext_string = isinstance(raw, str) and (
        "FULL-TEXT" in raw or "Introduction" in raw or "Materials and methods" in raw
    )
    has_pdf_object = isinstance(objects, list) and any(
        "pdf" in str(obj).lower() for obj in objects
    )

    full_text_available = has_fulltext_string or has_pdf_object

    if full_text_available:
        full_text = extract_text(raw).lower()
        matched_groups, group_hit_count, total_hits = score_relevance(full_text, keyword_groups)
        matched_keywords_text = flatten_matched_groups(matched_groups)
    else:
        matched_groups = {}
        group_hit_count = 0
        total_hits = 0
        matched_keywords_text = ""

    relevant_in_fulltext = (
        full_text_available and
        group_hit_count >= 2 and
        total_hits >= 3
    )

    results.append({
        "id_type": id_type,
        "id_value": id_value,
        "title": core.get("dc:title", rec["title"]),
        "doi": core.get("prism:doi", rec["doi"]),
        "openaccess": core.get("openaccess", ""),
        "full_text_available": full_text_available,
        "matched_keywords": matched_keywords_text,
        "group_hit_count": group_hit_count,
        "total_keyword_hits": total_hits,
        "relevant_in_fulltext": relevant_in_fulltext,
        "reason": "",
    })

    print(
        f"  -> done in {time.time() - item_start:.2f} sec | "
        f"full_text={full_text_available} | "
        f"group_hits={group_hit_count} | "
        f"total_hits={total_hits}"
    )

    if i % 20 == 0:
        pd.DataFrame(results).to_excel("fulltext_check_progress.xlsx", index=False)
        print(f"  Progress saved at record {i}")

results_df = pd.DataFrame(results)

selected_df = results_df[
    (results_df["full_text_available"] == True) &
    (results_df["relevant_in_fulltext"] == True)
].copy()

print("\nSummary")
print("Requested papers:", max_papers)
print("Collected from search:", len(search_df))
print("Checked with PII/DOI:", len(results_df))
print("Full-text papers:", results_df["full_text_available"].sum())
print("Relevant in full text:", len(selected_df))

# search_df.to_excel("search_results_1000.xlsx", index=False)
# results_df.to_excel("fulltext_and_keyword_check_1000.xlsx", index=False)
selected_df.to_excel("selected_relevant_fulltext_papers_1000.xlsx", index=False)
print(f"documents: {len(selected_df)}")

print("Saved files:")
# print("- search_results_1000.xlsx")
# print("- fulltext_and_keyword_check_1000.xlsx")
print("- selected_relevant_fulltext_papers_1000.xlsx")
# print("- fulltext_check_progress.xlsx")

Step 1: Starting paged search...
Requesting results 0 to 24...
Total available in Scopus: 19866
  Got 25 results, total collected: 25
Requesting results 25 to 49...
  Got 25 results, total collected: 50
Requesting results 50 to 74...
  Got 25 results, total collected: 75
Requesting results 75 to 99...
  Got 25 results, total collected: 100
Requesting results 100 to 124...
  Got 25 results, total collected: 125
Requesting results 125 to 149...
  Got 25 results, total collected: 150
Requesting results 150 to 174...
  Got 25 results, total collected: 175
Requesting results 175 to 199...
  Got 25 results, total collected: 200
Requesting results 200 to 224...
  Got 25 results, total collected: 225
Requesting results 225 to 249...
  Got 25 results, total collected: 250
Requesting results 250 to 274...
  Got 25 results, total collected: 275
Requesting results 275 to 299...
  Got 25 results, total collected: 300
Requesting results 300 to 324...
  Got 25 results, total collected: 325
Requesting

,title,pii,doi,eid,publication,coverDate
0,Advance analysis of aluminum alloy welded join...,,10.1007/s43939-025-00491-5,2-s2.0-105028455920,Discover Materials,2026-12-01
1,High power diode laser beam welding of AA8011 ...,,10.1038/s41598-026-41272-1,2-s2.0-105031448567,Scientific Reports,2026-12-01
2,Optimizing post-weld tempering to maximize for...,,10.1007/s41939-025-01151-0,2-s2.0-105029830871,Multiscale and Multidisciplinary Modeling Expe...,2026-12-01
3,Improving the Accuracy of Hardness Measurement...,,10.5829/ije.2026.39.10a.12,2-s2.0-105029546957,International Journal of Engineering Transacti...,2026-10-01
4,Microstructure and Wear Behavior of Repair Wel...,,10.17509/ijost.v11i2.89691,2-s2.0-105015306682,Indonesian Journal of Science and Technology,2026-09-01


Step 3: Preparing identifiers...
Total search results: 1000
Records with PII or DOI: 998
Step 4: Checking full text and content relevance...
[1/998] Checking DOI: 10.1007/s43939-025-00491-5
  -> read failed in 0.98 sec
[2/998] Checking DOI: 10.1038/s41598-026-41272-1
  -> read failed in 1.21 sec
[3/998] Checking DOI: 10.1007/s41939-025-01151-0
  -> read failed in 1.21 sec
[4/998] Checking DOI: 10.5829/ije.2026.39.10a.12
  -> read failed in 1.23 sec
[5/998] Checking DOI: 10.17509/ijost.v11i2.89691
  -> read failed in 1.22 sec
[6/998] Checking PII: S001623612600565X
  -> done in 1.56 sec | full_text=True | group_hits=2 | total_hits=7
[7/998] Checking PII: S0308016126000232
  -> done in 1.50 sec | full_text=True | group_hits=4 | total_hits=19
[8/998] Checking PII: S003039922600486X
  -> done in 1.56 sec | full_text=True | group_hits=4 | total_hits=14
[9/998] Checking DOI: 10.1142/S0219686726500319
  -> read failed in 1.16 sec
[10/998] Checking PII: S1005030225009478
  -> done in 1.90 sec 

In [22]:
apikey = "3af51ecbe651ef826516793ab9bbed05"

import re
import pandas as pd
from elsapy.elsclient import ElsClient
from elsapy.elsdoc import FullDoc

from llama_index.core import VectorStoreIndex, Document, Settings
from llama_index.embeddings.langchain import LangchainEmbedding
from langchain_huggingface import HuggingFaceEmbeddings
from llama_index.llms.ollama import Ollama

client = ElsClient(apikey)

input_file = "selected_relevant_fulltext_papers_1000.xlsx"
df = pd.read_excel(input_file).head(30)

def extract_text(raw_text):
    if isinstance(raw_text, str):
        return raw_text
    if isinstance(raw_text, dict):
        texts = []

        def recurse(x):
            if isinstance(x, str):
                texts.append(x)
            elif isinstance(x, dict):
                for v in x.values():
                    recurse(v)
            elif isinstance(x, list):
                for item in x:
                    recurse(item)

        recurse(raw_text)
        return " ".join(texts)
    return ""

def normalize_text(text):
    return re.sub(r"\s+", " ", text).strip()

def split_windows(text, window_size=1800, step=1200):
    text = normalize_text(text)
    if not text:
        return []

    if len(text) <= window_size:
        return [(0, text)]

    windows = []
    starts = list(range(0, len(text) - window_size + 1, step))
    if starts[-1] != len(text) - window_size:
        starts.append(len(text) - window_size)

    for start in starts:
        windows.append((start, text[start:start + window_size]))
    return windows

def bad_section(chunk):
    t = chunk.lower()
    bad_markers = [
        "references",
        "acknowledgement",
        "acknowledgment",
        "declaration of competing interest",
        "declarations",
        "credit authorship contribution statement",
        "funding",
        "data availability",
        "conflict of interest",
        "open access funding",
    ]
    return any(marker in t for marker in bad_markers)

section_keywords = {
    "methods_section": [
        "materials and methods",
        "experimental procedure",
        "experimental setup",
        "methodology",
        "sample preparation",
        "welding procedure",
        "process parameters",
        "experimental details",
        "materials",
        "methods"
    ],
    "equipment": [
        "welding machine",
        "power source",
        "torch",
        "thermocouple",
        "sem",
        "scanning electron microscope",
        "eds",
        "xrd",
        "microhardness",
        "hardness test",
        "tensile test",
        "tensile testing",
        "impact test",
        "charpy"
    ],
    "process": [
        "heat input",
        "cooling rate",
        "current",
        "voltage",
        "travel speed",
        "welding speed",
        "shielding gas",
        "interpass temperature",
        "preheat",
        "post weld heat treatment",
        "filler metal",
        "weld metal",
        "base metal",
        "gmaw",
        "gtaw",
        "saw",
        "fcaw"
    ],
    "outputs": [
        "yield strength",
        "ultimate tensile strength",
        "toughness",
        "microstructure",
        "hardness",
        "residual stress",
        "grain size",
        "phase transformation",
        "bainite",
        "martensite",
        "acicular ferrite",
        "carbides",
        "cct",
        "ttt"
    ]
}

def count_group_hits(text, keyword_dict):
    out = {}
    for group, kws in keyword_dict.items():
        hits = [kw for kw in kws if kw.lower() in text]
        out[group] = hits
    return out

def flatten_hits(hit_dict):
    parts = []
    for group, hits in hit_dict.items():
        if hits:
            parts.append(f"{group}: {', '.join(hits)}")
    return " | ".join(parts)

def chunk_quality_stats(chunk):
    t = chunk.lower()

    words = len(t.split())
    dots = t.count(".")
    commas = t.count(",")

    bad = (
        t.count("http") +
        t.count(".svg") +
        t.count(".jpg") +
        t.count(".jpeg") +
        t.count(".png") +
        t.count("altimg") * 2 +
        t.count("image/svg+xml") +
        t.count("image/jpeg") +
        t.count("image-high-res") +
        t.count("fig.") +
        t.count("table ") +
        t.count("doi") * 2 +
        t.count("references") * 3
    )

    citation_like = (
        t.count(" et al.") +
        t.count(" doi:") +
        t.count(" vol.") +
        t.count(" pp.")
    )

    digits = sum(c.isdigit() for c in t)
    letters = sum(c.isalpha() for c in t)
    digit_ratio = digits / max(len(t), 1)
    letter_ratio = letters / max(len(t), 1)

    return {
        "words": words,
        "dots": dots,
        "commas": commas,
        "bad": bad,
        "citation_like": citation_like,
        "digit_ratio": digit_ratio,
        "letter_ratio": letter_ratio,
    }

def score_chunk(chunk):
    t = chunk.lower()
    stats = chunk_quality_stats(chunk)

    if bad_section(chunk):
        return {
            "keep": False,
            "score": -999,
            "reason": "bad_section",
            "matched_groups": {},
            "group_hit_count": 0,
            "total_hits": 0,
        }

    matched_groups = count_group_hits(t, section_keywords)
    group_hit_count = sum(len(v) > 0 for v in matched_groups.values())
    total_hits = sum(len(v) for v in matched_groups.values())

    score = 0

    if stats["words"] >= 80:
        score += 1
    if stats["words"] >= 140:
        score += 1
    if stats["dots"] >= 3:
        score += 1
    if stats["commas"] >= 2:
        score += 1
    if stats["bad"] <= 6:
        score += 1
    if stats["citation_like"] <= 5:
        score += 1
    if stats["digit_ratio"] <= 0.25:
        score += 1
    if stats["letter_ratio"] >= 0.50:
        score += 1

    score += group_hit_count * 2
    score += min(total_hits, 6)

    if len(matched_groups["methods_section"]) > 0:
        score += 3
    if len(matched_groups["equipment"]) > 0:
        score += 2
    if len(matched_groups["process"]) > 0:
        score += 2
    if len(matched_groups["outputs"]) > 0:
        score += 1

    keep = (
        group_hit_count >= 2 and
        total_hits >= 3 and
        score >= 10
    )

    if len(matched_groups["methods_section"]) > 0 and len(matched_groups["process"]) > 0 and total_hits >= 3:
        keep = True

    return {
        "keep": keep,
        "score": score,
        "reason": "keep" if keep else "low_score_or_low_relevance",
        "matched_groups": matched_groups,
        "group_hit_count": group_hit_count,
        "total_hits": total_hits,
    }

documents = []
debug_rows = []

for i, row in df.iterrows():
    title = str(row["title"]).strip()
    id_type = str(row["id_type"]).strip().lower()
    id_value = str(row["id_value"]).strip()
    doi = str(row["doi"]).strip() if "doi" in row and pd.notna(row["doi"]) else ""

    print(f"[{i+1}/{len(df)}] Checking: {title}")

    try:
        doc = FullDoc(sd_pii=id_value) if id_type == "pii" else FullDoc(doi=id_value)
    except Exception as e:
        debug_rows.append({
            **row.to_dict(),
            "status": "drop",
            "reason": f"init_failed: {e}",
            "good_chunk_count": 0,
            "best_chunk_score": None,
            "best_chunk_hits": "",
            "preview": "",
        })
        print("  init failed")
        continue

    if not doc.read(client):
        debug_rows.append({
            **row.to_dict(),
            "status": "drop",
            "reason": "read_failed",
            "good_chunk_count": 0,
            "best_chunk_score": None,
            "best_chunk_hits": "",
            "preview": "",
        })
        print("  read failed")
        continue

    raw_text = doc.data.get("originalText", "")
    text = normalize_text(extract_text(raw_text))

    windows = split_windows(text, window_size=1800, step=1200)
    selected_chunks = []
    scored_chunks = []

    for start, chunk in windows:
        result = score_chunk(chunk)
        scored_chunks.append((start, chunk, result))
        if result["keep"]:
            selected_chunks.append((start, chunk, result))

    if selected_chunks:
        for j, (start, chunk, result) in enumerate(selected_chunks):
            documents.append(
                Document(
                    text=chunk,
                    metadata={
                        "title": title,
                        "id_type": id_type,
                        "id_value": id_value,
                        "doi": doi,
                        "chunk_start": start,
                        "chunk_id": j,
                        "group_hit_count": result["group_hit_count"],
                        "total_hits": result["total_hits"],
                        "matched_groups": flatten_hits(result["matched_groups"]),
                    }
                )
            )

        best_chunk = max(selected_chunks, key=lambda x: x[2]["score"])
        best_hits_text = flatten_hits(best_chunk[2]["matched_groups"])

        debug_rows.append({
            **row.to_dict(),
            "status": "keep",
            "reason": "has_relevant_chunks",
            "good_chunk_count": len(selected_chunks),
            "best_chunk_score": best_chunk[2]["score"],
            "best_chunk_hits": best_hits_text,
            "preview": best_chunk[1][:1000],
        })
        print(f"  keep | relevant chunks: {len(selected_chunks)}")
    else:
        if scored_chunks:
            best_chunk = max(scored_chunks, key=lambda x: x[2]["score"])
            best_hits_text = flatten_hits(best_chunk[2]["matched_groups"])
            preview_text = best_chunk[1][:1000]
            best_score = best_chunk[2]["score"]
            reason = best_chunk[2]["reason"]
        else:
            best_hits_text = ""
            preview_text = text[:1000]
            best_score = None
            reason = "no_windows"

        debug_rows.append({
            **row.to_dict(),
            "status": "drop",
            "reason": reason,
            "good_chunk_count": 0,
            "best_chunk_score": best_score,
            "best_chunk_hits": best_hits_text,
            "preview": preview_text,
        })
        print("  drop | no relevant chunks")

debug_df = pd.DataFrame(debug_rows)
debug_df.to_excel("chunk_filter_debug.xlsx", index=False)

print("\nTotal selected chunks:", len(documents))
print("Saved debug file: chunk_filter_debug.xlsx")

query_text = """
Find literature chunks that describe one welding-related experiment,
especially:
- experimental procedure or setup
- equipment used
- welding process parameters
- material system
- measurements or outputs
for welded steel or low alloy steel.
"""

if documents:
    Settings.embed_model = LangchainEmbedding(
        HuggingFaceEmbeddings(
            model_name="BAAI/bge-small-en-v1.5",
            encode_kwargs={"normalize_embeddings": True},
        )
    )
    Settings.llm = Ollama(model="llama3.2:3b", request_timeout=120.0)

    index = VectorStoreIndex.from_documents(documents)

    retriever = index.as_retriever(similarity_top_k=4)
    retrieved_nodes = retriever.retrieve(query_text)

    context_blocks = []

    for i, node in enumerate(retrieved_nodes, 1):
        title = node.metadata.get("title")
        doi = node.metadata.get("doi")
        chunk_start = node.metadata.get("chunk_start")
        matched_groups = node.metadata.get("matched_groups")
        chunk_text = node.text

        print(f"\n--- Retrieved chunk {i} ---")
        print("title:", title)
        print("doi:", doi)
        print("chunk_start:", chunk_start)
        print("matched_groups:", matched_groups)
        print(chunk_text)

        context_blocks.append(
            f"""[Retrieved Chunk {i}]
Title: {title}
DOI: {doi}
Chunk Start: {chunk_start}
Matched Groups: {matched_groups}

Text:
{chunk_text}"""
        )

    context_text = "\n\n" + ("\n\n" + "=" * 80 + "\n\n").join(context_blocks)

    prompt = f"""
Use ONLY the retrieved chunks below.

{context_text}

Task:
Identify one welding-related experiment described in the retrieved chunks
and summarize it in this format:

- Proposed Experiment:
- Needed Equipment:
- Key Process Parameters:
- Measurements / Outputs:
- Importance of this Experiment:
- Source Document Title:

Rules:
- Do not invent authors, years, citations, equipment, parameters, or source titles.
- Use only information explicitly supported by the retrieved chunks above.
- If the retrieved chunks are insufficient to specify equipment, procedure, parameters, or measurements, say so explicitly.
- Choose the experiment from ONLY ONE source document.
- Use the exact source document title from the retrieved chunk metadata.
- Do not combine information from multiple documents.
"""

    response = Settings.llm.complete(prompt)

    print("\nResponse:")
    print(response.text if hasattr(response, "text") else str(response))
else:
    print("\nNo relevant chunks found.")

[1/30] Checking: The effect of hydrogen odorisation with mercaptans on hydrogen uptake in L360NB pipeline steel: An experimental study
  keep | relevant chunks: 3
[2/30] Checking: Application of indentation test on strength characterization of the dissimilar metal welded joint
  keep | relevant chunks: 23
[3/30] Checking: Reduced ambient pressure and vacuum laser beam welding of FeCrAl cladding materials
  keep | relevant chunks: 9
[4/30] Checking: Inhibiting microstructure segregation and softening in a 700 MPa grade Al-Zn-Mg-Cu alloy by a novel underwater-assisted dual-rotation friction stir welding method
  keep | relevant chunks: 32
[5/30] Checking: Laser-modified wire arc cladding
  keep | relevant chunks: 19
[6/30] Checking: Multiphysics modeling of forced convection in narrow-gap gas metal arc welding: effects of stirring rod geometric configurations
  keep | relevant chunks: 9
[7/30] Checking: Ball indentation test: A versatile small-scale testing method for evaluating mechanic